In [1]:
from aalpy.learning_algs import run_Lstar
from aalpy.oracles import StatePrefixEqOracle
from aalpy.utils import save_automaton_to_file, visualize_automaton
import os
os.chdir("../scripts/extraction")

In [2]:
from DataProcessing import parse_data, preprocess_binary_classification_data
from RNN_SULs import RnnBinarySUL
from RNNClassifier import RNNClassifier

from TrainAndExtract import train_RNN_on_tomita_grammar, train_and_extract_bp, train_RNN_and_extract_FSM

[dynet] random seed: 1331741986
[dynet] allocating memory: 2048MB
[dynet] memory allocation done.


In [3]:
# Train the RNN on the data set generated from Tomita 3 Grammar
tomita_alphabet = ["0", "1"]
# load training data from file
x, y = parse_data("TrainingDataAndAutomata/tomita-3.txt")

# split test and validation data
x_train, y_train, x_test, y_test = preprocess_binary_classification_data(x, y, tomita_alphabet)

# Change parameters of the RNN if you want
rnn = RNNClassifier(tomita_alphabet, output_dim=2, num_layers=2, hidden_dim=50, batch_size=18,
                    x_train=x_train, y_train=y_train, x_test=x_test, y_test=y_test, nn_type="LSTM")


In [4]:
# Train the RNN
rnn.train(stop_acc=1.0, stop_loss=0.0005)

Starting train
Epoch 0: Accuracy 0.51393, Avg. Loss 12.47154 Validation Accuracy 0.50587
Epoch 1: Accuracy 0.5878, Avg. Loss 12.37014 Validation Accuracy 0.58803
Epoch 2: Accuracy 0.67106, Avg. Loss 11.88795 Validation Accuracy 0.67958
Epoch 3: Accuracy 0.87189, Avg. Loss 9.17798 Validation Accuracy 0.86502
Epoch 4: Accuracy 0.94371, Avg. Loss 4.29164 Validation Accuracy 0.93662
Epoch 5: Accuracy 0.96658, Avg. Loss 2.25509 Validation Accuracy 0.973
Epoch 6: Accuracy 0.99326, Avg. Loss 0.69929 Validation Accuracy 0.99531
Epoch 7: Accuracy 0.99971, Avg. Loss 0.20452 Validation Accuracy 1.0
Epoch 8: Accuracy 1.0, Avg. Loss 0.04526 Validation Accuracy 1.0
Epoch 9: Accuracy 1.0, Avg. Loss 0.02601 Validation Accuracy 1.0
Epoch 10: Accuracy 1.0, Avg. Loss 0.01821 Validation Accuracy 1.0
Done training!


In [5]:
# Wrap RNN in the SUL class. 
sul = RnnBinarySUL(rnn)
alphabet = tomita_alphabet

# Define the eq. oracle
state_eq_oracle = StatePrefixEqOracle(alphabet, sul, walks_per_state=200, walk_len=6)

In [6]:
# Extract the model from RNN
# Max. number of rounds is limited to be able to visualize small/correct automata.
# If it is not set adversarial inputs will be found :D 
dfa = run_Lstar(alphabet=alphabet, sul=sul, eq_oracle=state_eq_oracle, automaton_type='dfa',
                cache_and_non_det_check=True, max_learning_rounds=3)


Hypothesis 1: 1 states.
Hypothesis 2: 4 states.
Hypothesis 3: 5 states.
-----------------------------------
Learning Finished.
Learning Rounds:  3
Number of states: 5
Time (in seconds)
  Total                : 0.21
  Learning algorithm   : 0.01
  Conformance checking : 0.2
Learning Algorithm
 # Membership Queries  : 20
 # MQ Saved by Caching : 22
 # Steps               : 73
Equivalence Query
 # Membership Queries  : 1000
 # Steps               : 7799
-----------------------------------


In [7]:
save_automaton_to_file(dfa, f'RNN_Models/tomita{3}')
#visualize_automaton(dfa)

Model saved to RNN_Models/tomita3.dot.


In [ ]:
visualize_automaton(dfa)

Visualization started in the background thread.


Traceback (most recent call last):
  File "/home/yingshac/workspace/model_succ/venv/lib/python3.8/site-packages/pydot/core.py", line 1851, in create
    stdout_data, stderr_data, process = call_graphviz(
  File "/home/yingshac/workspace/model_succ/venv/lib/python3.8/site-packages/pydot/core.py", line 211, in call_graphviz
    process = subprocess.Popen(
  File "/usr/lib/python3.8/subprocess.py", line 858, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "/usr/lib/python3.8/subprocess.py", line 1704, in _execute_child
    raise child_exception_type(errno_num, err_msg, err_filename)
FileNotFoundError: [Errno 2] No such file or directory: 'dot'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/home/yingshac/workspace/model_succ/venv/lib/python3.8/site-packages/aalpy/utils/FileHandler.py", line 207, in save_automaton_to_file
    graph.write(path=path, format=file_type if file_type != 'dot' el

In [8]:
dfa.states

In [ ]:
#visualize_automaton(automaton, path="LearnedModel", file_type="pdf", display_same_state_trans=True):
print("#states = ", len(dfa.states))

import threading
visualization_thread = threading.Thread(target=save_automaton_to_file, name="Visualization",
                                            args=(dfa, "LearnedModel", "pdf", True, True, 2))
visualization_thread.start()

In [16]:
save_automaton_to_file(dfa, 
    path="LearnedModel", 
    file_type="pdf",
    display_same_state_trans=True, 
    visualize=False, 
    round_floats=2
)

Traceback (most recent call last):
  File "/home/yingshac/workspace/model_succ/venv/lib/python3.8/site-packages/pydot/core.py", line 1851, in create
    stdout_data, stderr_data, process = call_graphviz(
  File "/home/yingshac/workspace/model_succ/venv/lib/python3.8/site-packages/pydot/core.py", line 211, in call_graphviz
    process = subprocess.Popen(
  File "/usr/lib/python3.8/subprocess.py", line 858, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "/usr/lib/python3.8/subprocess.py", line 1704, in _execute_child
    raise child_exception_type(errno_num, err_msg, err_filename)
FileNotFoundError: [Errno 2] No such file or directory: 'dot'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/home/yingshac/workspace/model_succ/venv/lib/python3.8/site-packages/aalpy/utils/FileHandler.py", line 207, in save_automaton_to_file
    graph.write(path=path, format=file_type if file_type != 'dot' el